# Terjemahan Fable 5k ke Bahasa Melayu (Colab T4)

**Panduan ringkas**
1. `Runtime -> Change runtime type -> T4 GPU`.
2. Muat naik ke Google Drive dalam folder `MyDrive/xcoder-mt/`:
   - `fable_agent.jsonl` (dari repo: `datasets/fable_agent.jsonl`)
   - `import_fable.py` (dari repo: `datasets/scripts/import_fable.py`)
   (GGUF 4.7GB TIDAK perlu dimuat naik — sel 3 memuat turunnya terus dari Hugging Face)
3. Jalankan sel 1-6 berturut-turut, kemudian sel 7 (terjemahan penuh).
4. Jika sesi Colab terputus: sambung semula, jalankan semula sel 2-7.
   Cache di Drive disimpan setiap segmen siap, jadi kerja diteruskan dari titik terakhir.
5. Bila selesai: jalankan sel 8, muat turun `fable_agent_ms.jsonl` dari Drive.

Anggaran: ~2,405 segmen unik, ~3.1M aksara -> **~4-6 jam** pada T4. Sesi percuma biasanya ~4 jam.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

MT = '/content/drive/MyDrive/xcoder-mt'
WORK = '/content/work'
os.makedirs(WORK, exist_ok=True)
for name in ('fable_agent.jsonl', 'import_fable.py'):
    shutil.copy(f'{MT}/{name}', f'{WORK}/{name}')

if not os.path.exists(f'{WORK}/model.gguf'):
    path = hf_hub_download('unsloth/gemma-4-E4B-it-GGUF',
                           'gemma-4-E4B-it-Q4_K_M.gguf',
                           local_dir=WORK)
    os.replace(path, f'{WORK}/model.gguf')
print('model sedia:', os.path.getsize(f'{WORK}/model.gguf') / 1e9, 'GB')

In [ ]:
%pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

import glob, os, ctypes

cands = sorted(set(
    glob.glob('/usr/local/cuda*/lib64') +
    glob.glob('/usr/local/lib/python3*/dist-packages/nvidia/*/lib')))
for d in cands:
    os.environ['LD_LIBRARY_PATH'] = d + ':' + os.environ.get('LD_LIBRARY_PATH', '')
    for pat in ('libcudart.so*', 'libcublas.so*', 'libcublasLt.so*'):
        for so in glob.glob(f'{d}/{pat}'):
            try:
                ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
            except OSError:
                pass
print('lib dirs:', cands)
import llama_cpp
print('llama-cpp-python', llama_cpp.__version__)

In [ ]:
import subprocess, time, requests

srv = subprocess.Popen(
    ['python', '-m', 'llama_cpp.server',
     '--model', '/content/work/model.gguf', '--model_alias', 'gemma',
     '--n_gpu_layers', '-1', '--n_ctx', '8192',
     '--host', '127.0.0.1', '--port', '8080'],
    stdout=open('/content/work/server.log', 'w'), stderr=subprocess.STDOUT)

ok = False
for i in range(150):
    time.sleep(2)
    try:
        if requests.get('http://127.0.0.1:8080/v1/models', timeout=3).status_code == 200:
            ok = True
            print('server sedia selepas', (i + 1) * 2, 's')
            break
    except Exception:
        pass
if not ok:
    print('server gagal - lihat /content/work/server.log')
    print(open('/content/work/server.log').read()[-3000:])

In [ ]:
!python /content/work/import_fable.py translate \
  --in /content/work/fable_agent.jsonl \
  --out /content/drive/MyDrive/xcoder-mt/fable_agent_ms.jsonl \
  --server http://127.0.0.1:8080 --model gemma --workers 2 \
  --translate-output \
  --cache /content/drive/MyDrive/xcoder-mt/.fable_mt_cache.jsonl

In [ ]:
!python /content/work/import_fable.py clean \
  --in /content/drive/MyDrive/xcoder-mt/fable_agent_ms.jsonl \
  --out /content/drive/MyDrive/xcoder-mt/fable_agent_ms.jsonl
print('Siap - muat turun fable_agent_ms.jsonl dari Drive')